# Data Pipeline

## Problem Statement

Zepto's analysts need a way to benchmark catalog-style pricing and availability data before it ever reaches a dashboard.

**Objective:**
In this module, we scrape live product data from a public scraping-practice site, clean it, enrich it with the project's baseline fixed-rate currency conversion, and load it into a properly normalized relational database that we then query with both SQL and pandas — exactly the kind of raw-to-relational pipeline a catalog/competitive-intelligence workflow needs.


## Web Scraping Book Catalog Data

In this task, we scrape book data from the **Books to Scrape** website using **Requests** and **BeautifulSoup**. The collected raw dataset will be used in the subsequent stages of the pipeline for cleaning, transformation, database loading, and analysis.


In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin


In [2]:
BASE_URL = "https://books.toscrape.com/"
response = requests.get(BASE_URL)
print(f"Status Code: {response.status_code}")


Status Code: 200


In [3]:
soup = BeautifulSoup(response.content, 'html.parser')


### Extracting Book Categories


In [4]:
sidebar_categories = soup.find('div', class_='side_categories')
get_categories = sidebar_categories.find_all('a')

categories = []

for category in get_categories[1:]:
    category_name = category.text.strip()
    category_url = urljoin(BASE_URL, category['href'])
    categories.append({
        'name': category_name,
        'url': category_url
    })

print(f"Total Categories: {len(categories)}")


Total Categories: 50


In [5]:
print("Categories with urls:\n")
for i, category in enumerate(categories):
    print(f"Category {i+1}: {category['name']} - {category['url']}")


Categories with urls:

Category 1: Travel - https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Category 2: Mystery - https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Category 3: Historical Fiction - https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Category 4: Sequential Art - https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
Category 5: Classics - https://books.toscrape.com/catalogue/category/books/classics_6/index.html
Category 6: Philosophy - https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html
Category 7: Romance - https://books.toscrape.com/catalogue/category/books/romance_8/index.html
Category 8: Womens Fiction - https://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html
Category 9: Fiction - https://books.toscrape.com/catalogue/category/books/fiction_10/index.html
Category 10: Childrens - https://books.toscrape.com/catalogue/cat

### Collecting Book Links


In [6]:
def get_category_book_urls(category_url):
    book_urls = []
    current_page = category_url
    while current_page:
        response = requests.get(current_page)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        books = soup.find_all('article', class_='product_pod')

        for book in books:
            book_url = book.find('h3').find('a')['href']

            absolute_url = urljoin(current_page, book_url)
            book_urls.append(absolute_url)

        next_button = soup.find('li', class_='next')

        if next_button:
            next_page_url = next_button.find('a')['href']
            current_page = urljoin(current_page, next_page_url)
        else:
            current_page = None

    return book_urls
         

In [7]:
# travel_books = get_category_book_urls(categories[0]["url"])


In [8]:
print("Number of Books in Each Category:\n")
for i, category in enumerate(categories):
    print(f"{i+1} Category: {category['name']} --- No of books: {len(get_category_book_urls(category['url']))}")


Number of Books in Each Category:

1 Category: Travel --- No of books: 11
2 Category: Mystery --- No of books: 32
3 Category: Historical Fiction --- No of books: 26
4 Category: Sequential Art --- No of books: 75
5 Category: Classics --- No of books: 19
6 Category: Philosophy --- No of books: 11
7 Category: Romance --- No of books: 35
8 Category: Womens Fiction --- No of books: 17
9 Category: Fiction --- No of books: 65
10 Category: Childrens --- No of books: 29
11 Category: Religion --- No of books: 7
12 Category: Nonfiction --- No of books: 110
13 Category: Music --- No of books: 13
14 Category: Default --- No of books: 152
15 Category: Science Fiction --- No of books: 16
16 Category: Sports and Games --- No of books: 5
17 Category: Add a comment --- No of books: 67
18 Category: Fantasy --- No of books: 48
19 Category: New Adult --- No of books: 6
20 Category: Young Adult --- No of books: 54
21 Category: Science --- No of books: 14
22 Category: Poetry --- No of books: 19
23 Category: 

### Extracting Book Details


In [9]:
def get_book_details(book_url):
    response = requests.get(book_url)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, 'html.parser')

    title = soup.find('div', class_='product_main').find('h1').text.strip()

    price_gdp = soup.find('p', class_='price_color').text.strip()

    star_rating = soup.find('p', class_='star-rating')['class'][1]

    availability = soup.find('p', class_='instock availability').text.strip()

    breadcrumb = soup.find('ul', class_='breadcrumb').find_all('li')
    category = breadcrumb[2].text.strip()

    return {
        'title': title,
        'price_gdp': price_gdp,
        'star_rating': star_rating,
        'availability': availability,
        'category': category
    }
    

In [10]:
travel_books = get_category_book_urls(categories[0]["url"])
book = get_book_details(travel_books[0])
print(book)


{'title': "It's Only the Himalayas", 'price_gdp': '£45.17', 'star_rating': 'Two', 'availability': 'In stock (19 available)', 'category': 'Travel'}


### Scraping Selected Categories


In [11]:
selected_categories = ['Sequential Art', 'Fiction', 'Nonfiction', 'Young Adult']
all_books = []


In [12]:
category_url_map = {c['name']: c['url'] for c in categories}

for selected_category in selected_categories:
    category_url = category_url_map.get(selected_category)
    book_urls = get_category_book_urls(category_url)
    for book_url in book_urls:
        book = get_book_details(book_url)
        all_books.append(book)
        

In [13]:
print(f"Total number of books collected: {len(all_books)}")


Total number of books collected: 304
